# NSR 전사 서버 — 구글 콜랩판 (WhisperX)

폰 대신 콜랩의 GPU가 전사합니다. WhisperX 배치 추론이라 3시간 기록도 몇 분입니다.

**쓰는 법 (처음 3분)**

1. 위 메뉴 **런타임 → 런타임 유형 변경**에서 **T4 GPU** 를 고르십시오.
2. **런타임 → 모두 실행**을 누르십시오. 첫 실행은 모델을 받느라 3~6분 걸립니다.
3. 마지막 셀 출력에 나오는 **주소를 복사**해서, NSR 앱의
   **설정 → 전사 → 전사 서버·모델**의 **주소 칸**에 붙여넣으십시오.
4. **전사 모델은 앱에서 고릅니다** — 앱의 '전사 모델' 목록에서 누르면
   다음 전사부터 이 서버가 그 모델로 갈아끼워 돌립니다. 여기 '기본 모델' 셀의
   선택 상자는 앱이 아무것도 고르지 않았을 때만 쓰입니다.
5. (선택) **화자 분리**: 앱이 아니라 **콜랩에 한 번만** 준비합니다 —
   왼쪽 **열쇠 아이콘(보안 비밀)**에 `HF_TOKEN` 을 넣어 두면, 앱에서는
   화자 분리를 켜기만 하면 됩니다. 토큰 발급·모델 동의 안내는 아래 셀이
   실행될 때 출력됩니다.

이 판은 WhisperX 파이프라인입니다:
- **배치 추론** — 무음 기준으로 잘라 GPU에 묶음으로 넣어 수십 배 빠릅니다.
- **강제 정렬** — 전사 후 한국어 음소 정렬로 문장·단어 시각이 실측이 됩니다.
  앱에서 문장을 누르면 정확히 그 지점부터 재생됩니다.
- **단어 단위 화자 할당** — 화자 분리를 켜면 문장 중간에 화자가 바뀌어도
  경계가 맞습니다.

**알고 쓰십시오**

- 기록 음성 **원본이 구글(콜랩) 서버와 Cloudflare 터널을 지나갑니다.**
  내 컴퓨터가 아닙니다. 이 경로가 싫으면 내 컴퓨터 서버(앱의 '내 컴퓨터' 모드)를 쓰십시오.
- 주소 끝에 무작위 비밀 문자열이 붙어 있어서, 주소를 통째로 모르는 남은 못 씁니다.
  그래도 주소를 다른 곳에 붙여넣지 마십시오.
- 콜랩 화면에 **'런타임 연결이 끊겼습니다'가 떠도 전사는 계속되고 있을 수 있습니다** —
  폰의 진행률이 움직이면 서버는 살아 있는 것입니다. '재연결'은 눌러도 되지만,
  전사가 도는 동안 '모두 실행'을 다시 하지는 마십시오. 세션이 정말 회수되더라도
  앱은 그때까지 받은 부분을 저장해 두고, 기록은 다시 전사할 수 있게 남습니다.
- 콜랩 무료 세션은 탭을 닫거나 오래 놔두면 꺼집니다. 꺼졌으면 '모두 실행'을
  다시 — 주소가 새로 나오니 앱에도 다시 넣습니다.
- 전사할 때만 켜는 개인용입니다. 상시 서버로 두는 것은 콜랩 이용 규칙과 맞지 않습니다.
- 이 노트가 고쳐지면 **위 깃허브 링크로 새로 열어야** 최신판입니다.
  드라이브에 저장해 둔 사본은 옛 판 그대로입니다.


In [ ]:
# 필요한 것 설치 + 터널 프로그램 받기 (2~4분)
# whisperx 가 faster-whisper·pyannote.audio·정렬 모델 도구를 함께 끌고 온다.
# nvidia-cudnn/cublas 를 같이 까는 이유: 콜랩 기본 환경의 cuDNN 판이
# ctranslate2 와 어긋나면 첫 전사에서 파이썬이 통째로 죽는다("kernel
# restarted") — 실사용에서 그대로 재현된 사고다.
%pip -q install whisperx fastapi uvicorn python-multipart nvidia-cudnn-cu12 nvidia-cublas-cu12
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
print("설치 끝. 다음 셀로.")


In [ ]:
# ── 기본 모델 (앱이 고르지 않았을 때만) ──────────────────
# 모델은 보통 **앱의 '전사 모델' 목록에서** 고릅니다 — 앱이 전사 요청에
# 모델 id 를 실어 보내면 이 서버가 그 모델로 갈아끼워 돌립니다.
# 여기 선택 상자는 앱이 아무것도 고르지 않았을 때의 기본값입니다.
# 목록에 없는 공개 CT2 형식 모델 id 를 직접 붙여넣어도 됩니다.
#
#   NSR 한국어 Medium (대화 특화)       기본. 한국어 대화 1,273시간(소음 대화
#                                       포함)으로 재학습된 medium. 우리 저장소
#                                       에서 받는다(1.5GB, 램 안전, 허깅페이스
#                                       안 거침).
#   NSR 한국어 Large Turbo (fp16)        한국어 turbo(낭독 음성 학습)를 fp16 으로
#                                       줄인 판. 우리 저장소에서 받는다(1.5GB).
#   deepdml/...-large-v3-turbo-ct2      다국어 turbo. 빠르고 램 안전(1.6GB).
#   Systran/faster-whisper-large-v3     다국어 large. 제일 정확, 제일 느림(3GB).
#   Systran/faster-whisper-medium       다국어 medium. 가볍고 빠름.
MODEL_ID = "NSR 한국어 Medium (대화 특화)"  #@param ["NSR 한국어 Medium (대화 특화)", "NSR 한국어 Large Turbo (fp16)", "deepdml/faster-whisper-large-v3-turbo-ct2", "Systran/faster-whisper-large-v3", "Systran/faster-whisper-medium"] {allow-input: true}
print(f"이 세션의 기본 전사 모델: {MODEL_ID} (앱에서 고르면 그쪽이 우선합니다)")


In [ ]:
# 전사 서버 — 접수하고(202) 뒤에서 돌리고, 앱이 몇 초마다 결과를 물어간다.
#
# 왜 비동기인가: 다 될 때까지 한 요청으로 기다리는 방식은 앱의 업로드
# 클라이언트(60초)와 Cloudflare 터널(약 100초)이 먼저 끊는다 — 실기기
# 타임아웃으로 재현된 사실이다.
#
# WhisperX 파이프라인은 단계로 돈다: 전사(배치) → 정렬 → (선택) 화자 분리.
# 배치 추론은 도중 진행률이 없어서, 단계(stage)로 알리고 전사가 끝난 시점에
# 거친 세그먼트를 통째로 조회 응답(?since 증분)에 싣는다 — 세션이 도중에
# 회수돼도 앱은 받은 데까지 저장한다. 정렬·화자는 완성본(result)에 실린다.
import os
import threading
import tempfile
import traceback
import uuid

from fastapi import FastAPI, File, Form, UploadFile
from fastapi.responses import JSONResponse


def build_app(run_pipeline, secret: str, log=print) -> FastAPI:
    # log: 백그라운드 스레드의 print 는 셀이 끝난 뒤에는 콜랩 화면에 안
    # 보인다. 실행 셀이 큐를 비우며 대신 찍도록 콜백으로 받는다.
    app = FastAPI()
    jobs = {}
    gpu_lock = threading.Lock()  # GPU 는 하나 — 작업을 줄 세운다.

    def run_job(job_id, path, language, requested_model, diarize, hf_token):
        job = jobs[job_id]
        try:
            with gpu_lock:
                run_pipeline(job, path, language, requested_model, diarize, hf_token)
            log(f"전사 끝({job_id[:8]}): {len(job['segments'])}문장 / {round(job['result']['duration'])}초")
        except Exception:
            trace = traceback.format_exc()
            log(trace)
            job["error"] = trace[-1500:]
            job["status"] = "error"
        finally:
            os.unlink(path)

    @app.get(f"/{secret}/health")
    def health():
        return {"status": "ok"}

    @app.post(f"/{secret}/v1/audio/transcriptions")
    async def transcribe(
        file: UploadFile = File(...),
        language: str = Form("ko"),
        temperature: float = Form(0.0),
        prompt: str = Form(""),  # WhisperX 배치 경로는 프롬프트를 못 받는다 — 교정은 앱이 한다.
        response_format: str = Form("verbose_json"),
        model_name: str = Form("", alias="model"),  # 앱의 모델 선택 — 비면 세션 기본값
        diarize: str = Form(""),      # "1" 이면 화자 분리
        hf_token: str = Form(""),     # (예비) 앱이 보낸 토큰 — 보통은 콜랩 보안 비밀을 쓴다
    ):
        suffix = os.path.splitext(file.filename or "audio.m4a")[1] or ".m4a"
        with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as f:
            f.write(await file.read())
            path = f.name
        job_id = uuid.uuid4().hex
        jobs[job_id] = {"status": "queued", "progress": 0.0, "stage": "queued", "segments": []}
        threading.Thread(
            target=run_job,
            args=(job_id, path, language, model_name, diarize == "1", hf_token),
            daemon=True,
        ).start()
        log(
            f"전사 접수({job_id[:8]}): {file.filename}"
            + (f" · 모델 {model_name}" if model_name else "")
            + (" · 화자 분리" if diarize == "1" else "")
        )
        return JSONResponse(status_code=202, content={"job_id": job_id, "status": "queued"})

    @app.get(f"/{secret}/v1/audio/transcriptions/{{job_id}}")
    def job_status(job_id: str, since: int = 0):
        job = jobs.get(job_id)
        if job is None:
            return JSONResponse(
                status_code=404,
                content={"error": "모르는 작업입니다. 콜랩 세션이 재시작됐으면 전사를 다시 시작하십시오."},
            )
        if job["status"] == "done":
            return {"status": "done", "result": job["result"]}
        if job["status"] == "error":
            return {"status": "error", "error": job["error"]}
        done = job["segments"]
        return {
            "status": job["status"],
            "progress": round(job.get("progress", 0.0), 3),
            "stage": job.get("stage"),
            # since 이후의 새 세그먼트만 — 3초마다 물어도 응답이 가볍다.
            "segments": done[max(since, 0):],
            "next": len(done),
        }

    return app


In [ ]:
# 모델을 싣고 서버·터널을 띄운다. 마지막에 나오는 주소를 앱에 넣으면 된다.
import ctypes
import gc
import glob
import os
import re
import secrets
import subprocess
import tarfile
import threading
import time
import urllib.request

import numpy as np
import psutil

# cuDNN/cuBLAS 를 먼저 손으로 적재한다. 콜랩 기본 환경의 판과 어긋나면
# 첫 전사에서 파이썬이 통째로 죽는데("kernel restarted", 추적도 안 남는다),
# 방금 설치한 판을 절대 경로로 미리 올려 두면 이후 탐색이 이쪽을 쓴다.
for pattern in (
    "/usr/local/lib/python3*/dist-packages/nvidia/cublas/lib/libcublas*.so*",
    "/usr/local/lib/python3*/dist-packages/nvidia/cudnn/lib/libcudnn*.so*",
):
    for lib in sorted(glob.glob(pattern)):
        try:
            ctypes.CDLL(lib)
        except OSError:
            pass

import ctranslate2
import uvicorn
import whisperx
from huggingface_hub import snapshot_download

PORT = 8000
gpu = ctranslate2.get_cuda_device_count() > 0
DEVICE = "cuda" if gpu else "cpu"
BATCH = 16 if gpu else 4
if not gpu:
    print("⚠ GPU 가 안 잡혔습니다. 런타임 → 런타임 유형 변경 → T4 GPU 를 고른 뒤")
    print("  '모두 실행'을 다시 하십시오. CPU 로도 되지만 몇 배 느립니다.")

# ── 화자 분리 토큰 — 앱이 아니라 콜랩 보안 비밀에서 ─────────
# pyannote 모델은 무료지만 허깅페이스 게이트가 걸려 있다. 토큰을 앱에
# 넣게 하는 대신, 콜랩 왼쪽 열쇠(보안 비밀)에 HF_TOKEN 으로 한 번 넣어
# 두면 세션마다 자동으로 읽는다 — 앱에서는 화자 분리를 켜기만 하면 된다.
try:
    from google.colab import userdata
    HF_TOKEN = (userdata.get("HF_TOKEN") or "").strip()
except Exception:
    HF_TOKEN = ""
if HF_TOKEN:
    print("화자 분리 준비됨 — 콜랩 보안 비밀의 HF_TOKEN 을 씁니다.")
else:
    print("화자 분리를 쓰려면(선택, 한 번만):")
    print("  1) huggingface.co 무료 계정 → settings/tokens 에서 Read 토큰 발급")
    print("  2) 모델 동의 2회: huggingface.co/pyannote/speaker-diarization-3.1,")
    print("     huggingface.co/pyannote/segmentation-3.0")
    print("  3) 콜랩 왼쪽 열쇠 아이콘(보안 비밀)에 이름 HF_TOKEN 으로 저장,")
    print("     '노트북 액세스' 켜기 → '모두 실행' 다시.")

# ── 전사 모델 준비 — 앱이 고른 모델을 게으르게 싣고, 다르면 갈아끼운다 ──
# 지난 사고들을 여기서 계속 막는다.
#  · 401 사고: 없는 저장소에 익명 요청이 가면 허깅페이스는 404 대신 401 을
#    준다. token=False 로 못박아 낡은 토큰 개입도 잠갔다(공개 모델만 받는다).
#  · 램 사고: 모델을 갈아끼울 때 이전 모델을 먼저 내려놓고 받는다.
NSR_ID = "nsr-korean-medium"
NSR_TURBO_ID = "nsr-korean-large-turbo"
MIRROR_BASE = "https://github.com/lulus-cat/NSR-project/releases/download/models/"
MIRRORS = {
    NSR_ID: (MIRROR_BASE + "ct2-korean-medium-1273h-fp16.tar.gz",
             "/content/nsr-korean-medium", "한국어 Medium(대화 특화)"),
    NSR_TURBO_ID: (MIRROR_BASE + "ct2-korean-large-v3-turbo-fp16.tar.gz",
                   "/content/nsr-korean-large-turbo", "한국어 Large Turbo(fp16)"),
}


def canonical(mid):
    mid = (mid or "").strip()
    if not mid:
        return None
    if mid in MIRRORS:
        return mid
    if "Large Turbo" in mid or "large-turbo" in mid:
        return NSR_TURBO_ID
    if mid.startswith("NSR"):
        return NSR_ID
    return mid


def fetch_model_dir(mid):
    """모델 파일을 마련하고 (경로, compute_type) 을 돌려준다."""
    if mid in MIRRORS:
        url, model_dir, label = MIRRORS[mid]
        if not os.path.exists(os.path.join(model_dir, "model.bin")):
            say(f"{label}을 받는 중 — 약 1.4GB, 1~3분...")
            done = [-1]
            def _pct(count, block, total):
                pct = min(count * block * 100 // max(total, 1), 100)
                if pct // 10 > done[0]:
                    done[0] = pct // 10
                    say(f"  ...{pct}%")
            tar_path, _ = urllib.request.urlretrieve(url, reporthook=_pct)
            os.makedirs(model_dir, exist_ok=True)
            with tarfile.open(tar_path) as tar:
                tar.extractall(model_dir, filter="data")
            os.remove(tar_path)
        return model_dir, ("float16" if gpu else "int8")

    compute = "float16" if gpu else "int8"
    if "korean" in mid.lower():
        # float32 로 저장된 파인튜닝판 — 변환 없이 그대로("default") 싣는다.
        # 무료 콜랩 램(12.7GB)은 넘길 수 있다. 유료 고용량 램에서 쓰는 항목이다.
        compute = "default" if gpu else "int8"
        say("한국어 파인튜닝 float32 판: 램을 많이 씁니다 — 무료 콜랩이면 fp16 판을 쓰십시오.")
    say(f"모델 받는 중: {mid} (이 세션에서 처음 쓸 때만 내려받습니다)")
    try:
        model_dir = snapshot_download(
            mid,
            token=False,
            allow_patterns=["config.json", "preprocessor_config.json", "model.bin",
                            "tokenizer.json", "vocabulary.*"],
        )
    except Exception as e:
        raise RuntimeError(
            f"모델을 못 받았습니다: {mid} — id 철자를 확인하십시오"
            " (공개된 CT2 형식 모델이어야 합니다. 401 은 대부분 없는 저장소라는 뜻입니다)."
        ) from e
    return model_dir, compute


# 백그라운드 스레드의 print 는 셀이 끝나면 화면에 안 보인다. 큐에 쌓고
# 아래 상주 루프가 대신 찍는다 — 서버가 뜨기 전(초기 적재)에는 바로 찍는다.
events = []
serving = threading.Event()

def say(msg):
    if serving.is_set():
        events.append(msg)
    else:
        print(msg, flush=True)


_current = {"id": None, "model": None}
_model_lock = threading.Lock()


def get_model(requested=None):
    """앱이 요청한 모델의 WhisperX 배치 모델. 안 고르면 위 셀의 기본값."""
    mid = canonical(requested) or canonical(MODEL_ID) or NSR_ID
    with _model_lock:
        if _current["id"] == mid:
            return _current["model"]
        if _current["model"] is not None:
            say(f"모델 교체: {_current['id']} → {mid}")
            _current["model"] = None
            _current["id"] = None
            gc.collect()  # 이전 모델의 램·VRAM 을 먼저 돌려받는다.
        model_dir, compute = fetch_model_dir(mid)
        say(f"모델 여는 중: {mid}")
        model = whisperx.load_model(model_dir, DEVICE, compute_type=compute, language="ko")
        _current["id"] = mid
        _current["model"] = model
        return model


# 정렬(단어 시각) 모델 — 공개 한국어 wav2vec2 라 토큰이 필요 없다.
# 시작할 때 미리 실어 첫 전사가 정렬 다운로드를 기다리지 않게 한다.
print("정렬 모델 준비 중 (한국어 wav2vec2, 처음 한 번 1~2분)...")
ALIGN_MODEL, ALIGN_META = whisperx.load_align_model(language_code="ko", device=DEVICE)

# 화자 분리 파이프라인 — pyannote 3.1. 토큰이 있을 때만 게으르게 싣는다.
try:
    from whisperx.diarize import DiarizationPipeline
except Exception:  # 판에 따라 자리만 다르다
    DiarizationPipeline = whisperx.DiarizationPipeline

_diarizer = {"token": None, "pipe": None}
_diar_lock = threading.Lock()


def get_diarizer(token):
    with _diar_lock:
        if _diarizer["pipe"] is not None and _diarizer["token"] == token:
            return _diarizer["pipe"]
        say("화자 분리 모델 준비 중 (처음 한 번, 1~2분)...")
        try:
            pipe = DiarizationPipeline(use_auth_token=token, device=DEVICE)
        except Exception as e:
            raise RuntimeError(
                "화자 분리 모델을 못 실었습니다. 허깅페이스에서 두 모델"
                "(speaker-diarization-3.1, segmentation-3.0)에 '동의'했는지, "
                "콜랩 보안 비밀의 HF_TOKEN(Read)이 맞는지 확인하십시오."
            ) from e
        _diarizer["token"] = token
        _diarizer["pipe"] = pipe
        return pipe


def run_pipeline(job, path, language, requested_model, diarize, req_token):
    """WhisperX 3단: 배치 전사 → 강제 정렬 → (선택) 화자 분리."""
    job["status"] = "processing"
    job["stage"] = "model"
    model = get_model(requested_model)

    job["stage"] = "transcribe"
    audio = whisperx.load_audio(path)
    duration = round(len(audio) / 16000, 2)
    result = model.transcribe(audio, batch_size=BATCH, language=language or "ko")
    coarse = result["segments"]
    # 배치 전사는 도중 진행률이 없다 — 끝난 시점에 통째로 실어, 이후 단계에서
    # 세션이 회수돼도 앱이 여기까지는 건지게 한다.
    job["segments"].extend(
        {"id": i, "start": round(float(s["start"]), 2), "end": round(float(s["end"]), 2),
         "text": s["text"]}
        for i, s in enumerate(coarse)
    )
    job["progress"] = 0.7

    job["stage"] = "align"
    aligned = whisperx.align(coarse, ALIGN_MODEL, ALIGN_META, audio, DEVICE,
                             return_char_alignments=False)
    job["progress"] = 0.85

    token = (req_token or HF_TOKEN).strip()
    if diarize:
        if token:
            job["stage"] = "diarize"
            dia = get_diarizer(token)(audio)
            aligned = whisperx.assign_word_speakers(dia, aligned)
        else:
            say("화자 분리가 켜져 있지만 HF_TOKEN 이 없어 건너뜁니다 — 노트 위 안내를 보십시오.")

    final = []
    for i, s in enumerate(aligned["segments"]):
        start = s.get("start"); end = s.get("end")
        if start is None or end is None:  # 정렬이 못 잡은 짧은 조각은 원래 시각을 쓴다
            src = coarse[min(i, len(coarse) - 1)]
            start, end = src["start"], src["end"]
        seg = {"id": i, "start": round(float(start), 2), "end": round(float(end), 2),
               "text": s["text"]}
        if s.get("speaker"):
            seg["speaker"] = s["speaker"]
        final.append(seg)

    # 조회용 목록을 완성본으로 갈아끼운다(같은 리스트 객체를 유지해야
    # ?since 증분이 계속 동작한다).
    job["segments"][:] = final
    job["progress"] = 1.0
    job["result"] = {
        "task": "transcribe",
        "language": language or "ko",
        "duration": duration,
        "text": "".join(s["text"] for s in final).strip(),
        "segments": final,
    }
    job["status"] = "done"


# 자가 시험: 주소를 내주기 전에 기본 모델로 1초짜리 무음을 배치 전사해 본다.
# GPU 경로가 죽을 거라면 여기서 바로 죽어 원인이 이 셀에 보인다.
print("자가 전사 시험 중...")
get_model().transcribe(np.zeros(16000, dtype=np.float32), batch_size=4, language="ko")
print("자가 전사 시험 통과 — 전사 경로 정상.")
vm = psutil.virtual_memory()
print(f"메모리 {vm.used / 1e9:.1f} / {vm.total / 1e9:.1f} GB 사용 중 — 10GB 를 넘어가면 위험하다.")

secret = secrets.token_urlsafe(12)
app = build_app(run_pipeline, secret, log=say)
threading.Thread(
    target=lambda: uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning"),
    daemon=True,
).start()

tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
url = None
deadline = time.time() + 60
while time.time() < deadline and url is None:
    line = tunnel.stdout.readline()
    if not line and tunnel.poll() is not None:
        break
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line or "")
    if m:
        url = m.group(0)
if not url:
    raise RuntimeError("터널 주소를 못 받았습니다. 이 셀만 한 번 더 실행해 보십시오.")

print()
print("=" * 62)
print("NSR 앱에 넣을 주소 — 설정 → 전사 → 전사 서버·모델의 주소 칸:")
print()
print(f"    {url}/{secret}")
print()
print("전사 모델은 앱의 '전사 모델' 목록에서 고르십시오. 이 탭을 닫으면 서버도 꺼집니다.")
print("=" * 62)
print()
print("이 셀은 계속 실행 중인 것이 정상입니다 — 전사 접수/완료 로그가 아래에 찍힙니다.")
serving.set()
while True:
    time.sleep(2)
    while events:
        print(events.pop(0), flush=True)
